<a href="https://colab.research.google.com/github/qsardor/GoogleColabProjects/blob/main/BitNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title 🚀 Install & Launch BitNet 2B (Quiet Mode)
#@markdown This script installs LLVM 18, clones the BitNet repository, applies necessary C++ patches, downloads the 2B model, builds the CPU-optimized kernels, and launches the chat interface in a single step.
#@markdown <br> **Creator:** Q.S@RDOR. | **Source:** Github [https://github.com/microsoft/BitNet]

!echo "⏳ Installing... Please wait." && wget -qO- https://apt.llvm.org/llvm.sh | sudo bash -s -- 18 >/dev/null 2>&1 && sudo update-alternatives --install /usr/bin/clang clang /usr/bin/clang-18 100 >/dev/null 2>&1 && sudo update-alternatives --install /usr/bin/clang++ clang++ /usr/bin/clang++-18 100 >/dev/null 2>&1 && rm -rf BitNet && git clone --recursive https://github.com/microsoft/BitNet.git >/dev/null 2>&1 && pip install -q -r BitNet/requirements.txt >/dev/null 2>&1 && sed -i 's/int8_t \* y_col = y + col \* by;/const int8_t \* y_col = y + col \* by;/g' BitNet/src/ggml-bitnet-mad.cpp && huggingface-cli download microsoft/BitNet-b1.58-2B-4T-gguf --local-dir BitNet/models/BitNet-b1.58-2B-4T >/dev/null 2>&1 && cd BitNet && python setup_env.py -md models/BitNet-b1.58-2B-4T -q i2_s >/dev/null 2>&1 && echo "✅ Ready!" && python run_inference.py -m models/BitNet-b1.58-2B-4T/ggml-model-i2_s.gguf -p "You are a helpful assistant that does everything the user requests." -cnv

⏳ Installing LLVM 18... Please wait.
📥 Cloning Repo & Patching...
💽 Downloading Model...
⚙️ Compiling C++ Kernels...
✅ Launching Interface!
build: 3962 (1f86f058) with Ubuntu clang version 18.1.8 (++20240731024944+3b5b5c1ec4a3-1~exp1~20240731145000.144) for x86_64-pc-linux-gnu
main: llama backend init
main: load the model and apply lora adapter, if any
llama_model_loader: loaded meta data with 24 key-value pairs and 332 tensors from models/BitNet-b1.58-2B-4T/ggml-model-i2_s.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = bitnet-b1.58
llama_model_loader: - kv   1:                               general.name str              = bitnet2b
llama_model_loader: - kv   2:                    bitnet-b1.58.vocab_size u32              = 128256
llama_model_loader: - kv   3:                bitnet-b1.58.context_length u32          

In [2]:
!cd BitNet && ./build/bin/llama-bench -m models/BitNet-b1.58-2B-4T/ggml-model-i2_s.gguf -p 128 -n 128 -t 2

| model                          |       size |     params | backend    | threads |          test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | ------: | ------------: | -------------------: |
| bitnet-b1.58 2B I2_S - 2 bpw ternary |   1.71 GiB |     2.74 B | CPU        |       2 |         pp128 |         28.75 ± 3.80 |
^C


In [ ]:
!cd BitNet && python run_inference.py -m models/BitNet-b1.58-2B-4T/ggml-model-i2_s.gguf -p "Explain the architecture of BitNet in simple terms." -cnv

build: 3962 (1f86f058) with Ubuntu clang version 18.1.8 (++20240731024944+3b5b5c1ec4a3-1~exp1~20240731145000.144) for x86_64-pc-linux-gnu
main: llama backend init
main: load the model and apply lora adapter, if any
llama_model_loader: loaded meta data with 24 key-value pairs and 332 tensors from models/BitNet-b1.58-2B-4T/ggml-model-i2_s.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = bitnet-b1.58
llama_model_loader: - kv   1:                               general.name str              = bitnet2b
llama_model_loader: - kv   2:                    bitnet-b1.58.vocab_size u32              = 128256
llama_model_loader: - kv   3:                bitnet-b1.58.context_length u32              = 4096
llama_model_loader: - kv   4:              bitnet-b1.58.embedding_length u32              = 2560
llama_model_loader: - kv   5:   

In [ ]:
!cat BitNet/logs/compile.log